# Model training

This stage involves training the ML algorithm by providing it with datasets, where the learning process takes place. Consistent training can significantly enhance the model's prediction accuracy. It's essential to initialize the model's weights randomly so the algorithm can effectively learn to adjust them.

# XGBoost

In [ ]:
from xgboost import XGBRFClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from scipy.stats import randint

model = XGBRFClassifier(random_state=42)
params = {
    "learning_rate": uniform(1e-2, 3e-1),
    "min_split_loss": uniform(0, 10),
    "max_depth": randint(3, 10),
    "subsample": uniform(0, 1),
    "objective": ["reg:squarederror", "binary:logistic", "reg:logistic"],
    "eval_metric": ["aucpr", "error"]
}

model_grid = RandomizedSearchCV(model, param_distributions=params, n_jobs=-1, verbose=3, n_iter=10, cv=10)

model_grid.fit(X_train, y_train)

# Model test accuracy

In [ ]:
from sklearn.metrics import accuracy_score

best_model_xgboost_params = model_grid.best_params_
print("Best xgboost params")
pprint(best_model_xgboost_params)

y_pred_train = model_grid.predict(X_train)
y_pred_test = model_grid.predict(X_test)
print("Accuracy train", accuracy_score(y_pred_train, y_train ))
print("Accuracy test", accuracy_score(y_pred_test, y_test))


# XGBoost performance overview
* Confusion matrix
* Classification report

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

conf_matrix = confusion_matrix(y_test, y_pred_test)
print("Test actual/predicted\n")
print(pd.crosstab(y_test, y_pred_test, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_test, y_pred_test),'\n')

conf_matrix = confusion_matrix(y_train, y_pred_train)
print("Train actual/predicted\n")
print(pd.crosstab(y_train, y_pred_train, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_train, y_pred_train),'\n')

# Save best XGBoost model

In [ ]:
xgboost_model = model_grid.best_estimator_
xgboost_model_path = "./artifacts/lead_model_xgboost.json"
xgboost_model.save_model(xgboost_model_path)

model_results = {
    xgboost_model_path: classification_report(y_train, y_pred_train, output_dict=True)
}

# SKLearn logistic regression

In [ ]:
import mlflow.pyfunc

from sklearn.linear_model import LogisticRegression
import os
from sklearn.metrics import cohen_kappa_score, f1_score
import matplotlib.pyplot as plt
import joblib

class lr_wrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model):
        self.model = model
    
    def predict(self, context, model_input):
        return self.model.predict_proba(model_input)[:, 1]


mlflow.sklearn.autolog(log_input_examples=True, log_models=False)
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

with mlflow.start_run(experiment_id=experiment_id) as run:
    model = LogisticRegression()
    lr_model_path = "./artifacts/lead_model_lr.pkl"

    params = {
              'solver': ["newton-cg", "lbfgs", "liblinear", "sag", "saga"],
              'penalty':  ["none", "l1", "l2", "elasticnet"],
              'C' : [100, 10, 1.0, 0.1, 0.01]
    }
    model_grid = RandomizedSearchCV(model, param_distributions= params, verbose=3, n_iter=10, cv=3)
    model_grid.fit(X_train, y_train)

    best_model = model_grid.best_estimator_

    y_pred_train = model_grid.predict(X_train)
    y_pred_test = model_grid.predict(X_test)


    # log artifacts
    mlflow.log_metric('f1_score', f1_score(y_test, y_pred_test))
    mlflow.log_artifacts("artifacts", artifact_path="model")
    mlflow.log_param("data_version", "00000")
    
    # store model for model interpretability
    joblib.dump(value=model, filename=lr_model_path)
        
    # Custom python model for predicting probability 
    mlflow.pyfunc.log_model('model', python_model=lr_wrapper(model))


model_classification_report = classification_report(y_test, y_pred_test, output_dict=True)

best_model_lr_params = model_grid.best_params_

print("Best lr params")
pprint(best_model_lr_params)

print("Accuracy train:", accuracy_score(y_pred_train, y_train ))
print("Accuracy test:", accuracy_score(y_pred_test, y_test))

conf_matrix = confusion_matrix(y_test, y_pred_test)
print("Test actual/predicted\n")
print(pd.crosstab(y_test, y_pred_test, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_test, y_pred_test),'\n')

conf_matrix = confusion_matrix(y_train, y_pred_train)
print("Train actual/predicted\n")
print(pd.crosstab(y_train, y_pred_train, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_train, y_pred_train),'\n')

model_results[lr_model_path] = model_classification_report
print(model_classification_report["weighted avg"]["f1-score"])


# Save columns and model results

In [ ]:
column_list_path = './artifacts/columns_list.json'
with open(column_list_path, 'w+') as columns_file:
    columns = {'column_names': list(X_train.columns)}
    pprint(columns)
    json.dump(columns, columns_file)

print('Saved column list to ', column_list_path)

model_results_path = "./artifacts/model_results.json"
with open(model_results_path, 'w+') as results_file:
    json.dump(model_results, results_file)

# New code

In [1]:
max_date = "2024-01-31"
min_date = "2024-01-01"

In [2]:
import os
import shutil
from pprint import pprint

# shutil.rmtree("./artifacts",ignore_errors=True)
os.makedirs("artifacts",exist_ok=True)
print("Created artifacts directory")

Created artifacts directory


In [3]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format',lambda x: "%.3f" % x)

In [4]:
from xgboost import XGBRFClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from scipy.stats import randint
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import mlflow.pyfunc
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score
import matplotlib.pyplot as plt
import joblib
import datetime
import json
import numpy as np

In [5]:
dataX = pd.read_csv("./artifacts/X_test.csv")
datay = pd.read_csv("./artifacts/y_test.csv")

display(dataX.head(5))
display(datay.head(5))

,purchases,time_spent,n_visits,customer_group_2,customer_group_3,customer_group_4,customer_group_5,customer_group_6,customer_group_7,customer_group_8,customer_group_9,onboarding_True
0,0.730,0.153,0.239,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000
1,0.294,0.482,0.479,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,0.294,0.469,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000
3,0.621,0.392,0.419,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,0.076,0.837,0.120,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000


,lead_indicator
0,0.000
1,1.000
2,0.000
3,1.000
4,0.000


In [6]:
#from sklearn.model_selection import train_test_split

#X_train, X_test, y_train, y_test = train_test_split(
#    X, y, random_state=42, test_size=0.15, stratify=y
#)
#y_train

X_train=dataX
y_train=datay
X_test=dataX
y_test=datay

# Start experiment

In [7]:
current_date = datetime.datetime.now().strftime("%Y_%B_%d")
artifact_path = "model"
model_name = "lead_model"
experiment_name = current_date

mlflow.set_experiment(experiment_name)
mlflow.sklearn.autolog(log_input_examples=True, log_models=False)
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
experiment_id


2025/11/21 12:13:43 INFO mlflow.tracking.fluent: Experiment with name '2025_November_21' does not exist. Creating a new experiment.


'570933180417166162'

In [8]:
mlflow.set_tracking_uri()
mlflow.get_tracking_uri()

TypeError: set_tracking_uri() missing 1 required positional argument: 'uri'

In [18]:
!mlflow ui --port 5000 

[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [4575]
INFO:     Started server process [4579]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Started server process [4577]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Started server process [4578]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Started server process [4580]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     127.0.0.1:38954 - "GET / HTTP/1.1" 304 Not Modified
INFO:     127.0.0.1:38954 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=5&order_by=last_update_time+DESC HTTP/1.1" 200 OK
INFO

In [10]:
class XGBWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model):
        self.model = model
    
    def predict(self, context, model_input):
        return self.model.predict_proba(model_input)[:, 1]

/home/dimiko/MLOps/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [11]:
with mlflow.start_run(experiment_id=experiment_id) as run:
    model = XGBRFClassifier(random_state=42)
    xgb_model_path = "./artifacts/lead_model_xgb.pkl"

    params = {
        "learning_rate": uniform(1e-2, 3e-1),
        "min_split_loss": uniform(0, 10),
        "max_depth": randint(3, 10),
        "subsample": uniform(0, 1),
        "objective": ["reg:squarederror", "binary:logistic", "reg:logistic"],
        "eval_metric": ["aucpr", "error"]
    }
    model_grid = RandomizedSearchCV(model, param_distributions=params, n_jobs=-1, verbose=3, n_iter=10, cv=10)

    model_grid.fit(X_train, y_train)

    best_model = model_grid.best_estimator_

    y_pred_train = model_grid.predict(X_train)
    y_pred_test = model_grid.predict(X_test)

    #print("Best xgboost params")
    #pprint(model_grid.best_params_)
    #print("Accuracy train", accuracy_score(y_pred_train, y_train ))
    #print("Accuracy test", accuracy_score(y_pred_test, y_test))
    

    # log artifacts
    mlflow.log_metric('f1_score', f1_score(y_test, y_pred_test))
    mlflow.log_artifacts("artifacts", artifact_path="model")
    mlflow.log_param("data_version", "00000")
    
    # store model for model interpretability
    joblib.dump(value=model, filename=xgb_model_path)
        
    # Custom python model for predicting probability 
    mlflow.pyfunc.log_model('model', python_model=XGBWrapper(model))

Fitting 10 folds for each of 10 candidates, totalling 100 fits
[CV 1/10] END eval_metric=aucpr, learning_rate=0.2393685635010706, max_depth=7, min_split_loss=9.161097487949691, objective=reg:logistic, subsample=0.1303108919184779;, score=0.500 total time=   2.5s
[CV 1/10] END eval_metric=aucpr, learning_rate=0.2853211494378873, max_depth=5, min_split_loss=6.826891101326179, objective=binary:logistic, subsample=0.7694148301685814;, score=0.750 total time=   2.5s
[CV 2/10] END eval_metric=aucpr, learning_rate=0.2853211494378873, max_depth=5, min_split_loss=6.826891101326179, objective=binary:logistic, subsample=0.7694148301685814;, score=0.750 total time=   2.7s
[CV 3/10] END eval_metric=aucpr, learning_rate=0.2393685635010706, max_depth=7, min_split_loss=9.161097487949691, objective=reg:logistic, subsample=0.1303108919184779;, score=0.500 total time=   0.0s
[CV 4/10] END eval_metric=aucpr, learning_rate=0.2393685635010706, max_depth=7, min_split_loss=9.161097487949691, objective=reg:log

2025/11/21 12:17:42 INFO mlflow.sklearn.utils: Logging the 5 best runs, 5 runs will be omitted.
2025/11/21 12:17:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/21 12:17:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [12]:
class lr_wrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model):
        self.model = model
    
    def predict(self, context, model_input):
        return self.model.predict_proba(model_input)[:, 1]


/home/dimiko/MLOps/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [13]:
with mlflow.start_run(experiment_id=experiment_id) as run:
    model = LogisticRegression()
    lr_model_path = "./artifacts/lead_model_lr.pkl"

    params = {
              'solver': ["newton-cg", "lbfgs", "liblinear", "sag", "saga"],
              'penalty':  ["none", "l1", "l2", "elasticnet"],
              'C' : [100, 10, 1.0, 0.1, 0.01]
    }
    model_grid = RandomizedSearchCV(model, param_distributions= params, verbose=3, n_iter=10, cv=3)
    model_grid.fit(X_train, y_train)

    best_model = model_grid.best_estimator_

    y_pred_train = model_grid.predict(X_train)
    y_pred_test = model_grid.predict(X_test)

    print("Best lr params")
    pprint(model_grid.best_params_)
    print("Accuracy train", accuracy_score(y_pred_train, y_train ))
    print("Accuracy test", accuracy_score(y_pred_test, y_test))
    # log artifacts
    mlflow.log_metric('f1_score', f1_score(y_test, y_pred_test))
    mlflow.log_artifacts("artifacts", artifact_path="model")
    mlflow.log_param("data_version", "00000")
    
    # store model for model interpretability
    joblib.dump(value=model, filename=lr_model_path)
        
    # Custom python model for predicting probability 
    mlflow.pyfunc.log_model('model', python_model=lr_wrapper(model))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV 1/3] END C=0.01, penalty=l1, solver=newton-cg;, score=nan total time=   0.0s
[CV 2/3] END C=0.01, penalty=l1, solver=newton-cg;, score=nan total time=   0.0s
[CV 3/3] END C=0.01, penalty=l1, solver=newton-cg;, score=nan total time=   0.0s
[CV 1/3] END ....C=100, penalty=l1, solver=saga;, score=0.833 total time=   0.0s
[CV 2/3] END ....C=100, penalty=l1, solver=saga;, score=0.750 total time=   0.0s
[CV 3/3] END ....C=100, penalty=l1, solver=saga;, score=0.417 total time=   0.0s
[CV 1/3] END .....C=0.1, penalty=l2, solver=sag;, score=0.750 total time=   0.0s
[CV 2/3] END .....C=0.1, penalty=l2, solver=sag;, score=0.583 total time=   0.0s
[CV 3/3] END .....C=0.1, penalty=l2, solver=sag;, score=0.667 total time=   0.0s
[CV 1/3] END .....C=10, penalty=l2, solver=saga;, score=0.667 total time=   0.0s
[CV 2/3] END .....C=10, penalty=l2, solver=saga;, score=0.750 total time=   0.0s
[CV 3/3] END .....C=10, penalty=l2, solver=saga;

2025/11/21 12:18:05 INFO mlflow.sklearn.utils: Logging the 5 best runs, 5 runs will be omitted.


Best lr params
{'C': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Accuracy train 0.8611111111111112
Accuracy test 0.8611111111111112


2025/11/21 12:18:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/21 12:18:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [14]:
from mlflow.tracking import MlflowClient
client=MlflowClient()
client.search_experiments()

[<Experiment: artifact_location='file:///mnt/c/Users/user/Desktop/school/ITU/Sem5/DSP/itu-MLOps-project/project/project_repo/notebooks/mlruns/570933180417166162', creation_time=1763723623224, experiment_id='570933180417166162', last_update_time=1763723623224, lifecycle_stage='active', name='2025_November_21', tags={'mlflow.experimentKind': 'custom_model_development'}>,
 <Experiment: artifact_location='file:///mnt/c/Users/user/Desktop/school/ITU/Sem5/DSP/itu-MLOps-project/project/project_repo/notebooks/mlruns/864364772299070208', creation_time=1762763665766, experiment_id='864364772299070208', last_update_time=1762763665766, lifecycle_stage='active', name='2025_November_10', tags={'mlflow.experimentKind': 'custom_model_development'}>,
 <Experiment: artifact_location='file:///mnt/c/Users/user/Desktop/school/ITU/Sem5/DSP/itu-MLOps-project/project/project_repo/notebooks/mlruns/0', creation_time=1762763665599, experiment_id='0', last_update_time=1762763665599, lifecycle_stage='active', name

# Evaluate

In [15]:
model_results = {
    xgb_model_path: classification_report(y_train, y_pred_train, output_dict=True)
}

In [16]:
model_classification_report = classification_report(y_test, y_pred_test, output_dict=True)

best_model_lr_params = model_grid.best_params_

print("Best lr params")
pprint(best_model_lr_params)

print("Accuracy train:", accuracy_score(y_pred_train, y_train ))
print("Accuracy test:", accuracy_score(y_pred_test, y_test))

conf_matrix = confusion_matrix(y_test, y_pred_test)
y_test = np.ravel(y_test)
y_pred_test = np.ravel(y_pred_test)
y_train = np.ravel(y_train)
y_pred_train = np.ravel(y_pred_train)
print("Test actual/predicted\n")
print(pd.crosstab(y_test, y_pred_test, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_test, y_pred_test),'\n')

conf_matrix = confusion_matrix(y_train, y_pred_train)
print("Train actual/predicted\n")
print(pd.crosstab(y_train, y_pred_train, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_train, y_pred_train),'\n')

model_results[lr_model_path] = model_classification_report
print(model_classification_report["weighted avg"]["f1-score"])

Best lr params
{'C': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Accuracy train: 0.8611111111111112
Accuracy test: 0.8611111111111112
Test actual/predicted

Predicted  0.000  1.000   All
Actual                       
0.000         16      2    18
1.000          3     15    18
 All          19     17    36 

Classification report

              precision    recall  f1-score   support

         0.0       0.84      0.89      0.86        18
         1.0       0.88      0.83      0.86        18

    accuracy                           0.86        36
   macro avg       0.86      0.86      0.86        36
weighted avg       0.86      0.86      0.86        36
 

Train actual/predicted

Predicted  0.000  1.000   All
Actual                       
0.000         16      2    18
1.000          3     15    18
 All          19     17    36 

Classification report

              precision    recall  f1-score   support

         0.0       0.84      0.89      0.86        18
         1.0       0.88      0.83 

In [17]:
column_list_path = './artifacts/columns_list.json'
with open(column_list_path, 'w+') as columns_file:
    columns = {'column_names': list(X_train.columns)}
    pprint(columns)
    json.dump(columns, columns_file)

print('Saved column list to ', column_list_path)

model_results_path = "./artifacts/model_results.json"
with open(model_results_path, 'w+') as results_file:
    json.dump(model_results, results_file)

{'column_names': ['purchases',
                  'time_spent',
                  'n_visits',
                  'customer_group_2',
                  'customer_group_3',
                  'customer_group_4',
                  'customer_group_5',
                  'customer_group_6',
                  'customer_group_7',
                  'customer_group_8',
                  'customer_group_9',
                  'onboarding_True']}
Saved column list to  ./artifacts/columns_list.json
